# 0. Extract: 제7차 작업환경 실태조사 → 표준화 CSV
- **입력**: `Data_kosha/작업환경 실태조사/7차.../Dataset1.csv` (N=20,262)
- **출력**: `output/pre_output/wes7_raw.csv`
- **Portable**: Colab + Local 자동 감지
- **Fix (Colab)**: Google Drive sync race 방지 — SHA를 메모리에서 계산

In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 1 — Extract, Clean, Save + Auto Methods (Section 2.1-2.2)
# ══════════════════════════════════════════════════════════════
# ── Portable path setup (Colab + Local) ──
import os, sys
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/완석_구글자료/연구자료/20260313_kosha'
except ImportError:
    BASE = '/Users/y3korea/Library/CloudStorage/GoogleDrive-y3korea@gmail.com/내 드라이브/완석_구글자료/연구자료/20260313_kosha'
assert os.path.exists(BASE), f'BASE not found: {BASE}'
print(f'BASE: {BASE}')

import pandas as pd, numpy as np, hashlib

DATA_DIR = os.path.join(BASE, 'Data_kosha', '작업환경 실태조사', '7차_작업환경 실태조사',
                        '[CSV] 제7차 작업환경실태조사 데이터(CSV)')
OUT_DIR  = os.path.join(BASE, 'Code_kosha', '2_code', 'output', 'pre_output')
PAPER_DIR = os.path.join(BASE, 'Code_kosha', '2_code', 'paper', 'auto')
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(PAPER_DIR, exist_ok=True)

# ── Variable definitions ──
ID_VARS = ['id', 'year', 'ksic2', 'wt1', 'wt2']
SC_DIMS = {
    'A_mgmt':  ['mgt_emph_saf', 'mgt_prior_saf', 'mgt_value_saf'],
    'B_comm':  ['saf_disc_opp', 'saf_open_disc', 'saf_feed_reg', 'saf_sug_sys', 'saf_sug_resp'],
    'C_train': ['saf_tr_opp', 'saf_tr_effect'],
    'D_sys':   ['saf_sys_proc', 'saf_proc_effect', 'saf_equip_avail'],
    'E_empow': ['work_ref_unsaf', 'work_vol_saf'],
}
SC_ALL = [i for items in SC_DIMS.values() for i in items]  # 15 items
SAF_STRUCT = ['saf_com_yn', 'saf_dept_yn', 'saf_mgr_yn', 'saf_rep_yn', 'saf_sup_yn',
              'saf_work_pct', 'sup_prev_acc']
VIC_VARS = []
for y in [2022, 2023, 2024]:
    for p in ['vic_', 'acc_vic_', 'ill_vic_', 'dth_', 'acc_dth_', 'ill_dth_']:
        for k in ['occ', 'appr']:
            VIC_VARS.append(f'{p}{y}_{k}')
DEMO_VARS = ['r_wrk_tot','r_eld_tot','r_frgn_tot','r_fem_tot','r_dis_tot','ovt_yn','ovt_pct','shift_yn']
ALL_VARS = ID_VARS + SC_ALL + SAF_STRUCT + VIC_VARS + DEMO_VARS

# ── Load ──
fp = os.path.join(DATA_DIR, 'Dataset1.csv')
df_raw = pd.read_csv(fp, encoding='utf-8-sig', low_memory=False)
N_raw = len(df_raw)
print(f'Raw load: N={N_raw:,}, cols={df_raw.shape[1]}')

df = df_raw[[v for v in ALL_VARS if v in df_raw.columns]].copy()
# Preserve 'id' as string (format: "100-011") — avoid NaN from numeric coercion
ID_COL_PRESERVE = ['id']
for c in df.columns:
    if c in ID_COL_PRESERVE:
        df[c] = df[c].astype(str)
    else:
        df[c] = pd.to_numeric(df[c], errors='coerce')

# ── Missing code handling ──
for v in SC_ALL:
    df.loc[~df[v].isin([1,2,3,4,5]), v] = np.nan
for v in ['saf_mgr_yn','saf_rep_yn','saf_sup_yn','saf_com_yn','saf_dept_yn']:
    if v in df.columns:
        df.loc[df[v].isin([3,4]), v] = np.nan
for v in ['sup_prev_acc','ovt_yn','shift_yn']:
    if v in df.columns:
        df.loc[~df[v].isin([1,2]), v] = np.nan
for v in VIC_VARS:
    if v in df.columns:
        df.loc[df[v] < 0, v] = np.nan
        df[v] = df[v].fillna(0).astype(float)
for v in ['r_wrk_tot','r_eld_tot','r_frgn_tot','r_fem_tot','r_dis_tot']:
    if v in df.columns:
        df.loc[~df[v].isin(range(1,10)), v] = np.nan

# ── Save CSV (Colab-safe: compute SHA from memory, avoid Drive sync race) ──
out_fp = os.path.join(OUT_DIR, 'wes7_raw.csv')
csv_content = df.to_csv(index=False)
sha = hashlib.sha256(csv_content.encode('utf-8')).hexdigest()[:16]
with open(out_fp, 'w', encoding='utf-8-sig') as f:
    f.write(csv_content)

# ── Verification printout ──
n_ind = int(df['ksic2'].nunique())
vic_24_sum = int(df['vic_2024_appr'].sum())
vic_24_zero_pct = (df['vic_2024_appr']==0).mean()*100
print(f'\n=== EXTRACT RESULT ===')
print(f'N={len(df):,}, cols={df.shape[1]}, SHA={sha}')
print(f'Industries (KSIC 2-digit): {n_ind}')
print(f'SC items non-null: {df[SC_ALL].notna().mean().mean()*100:.1f}%')
print(f'Accident outcomes (2024 approved): sum={vic_24_sum}, zero%={vic_24_zero_pct:.1f}')

# ══════════════════════════════════════════════════════════════
# AUTO-GENERATE METHODS SECTIONS 2.1–2.2 (English, Safety Science style)
# ══════════════════════════════════════════════════════════════

methods_md = f"""# Methods (auto-generated from 0_extract.ipynb)

## 2.1 Study Design and Data Source

We conducted a cross-sectional analysis of the Seventh Korean Working Environment
Survey (WES-7), a nationally representative establishment-level survey administered
in 2024 by the Korea Occupational Safety and Health Research Institute (KOSHRI).
The WES targets workplaces with at least one employee across all industrial
sectors defined by the Korean Standard Industrial Classification (KSIC). The
sampling frame is stratified by industry (KSIC 2-digit level) and establishment
size, with probability-proportional-to-size selection within strata. Survey weights
(wt1) project the sample to the national population of Korean establishments. Data
were collected through face-to-face interviews with employer representatives
(safety managers or owners) and cover safety climate, ergonomic exposures,
workforce composition, working-time arrangements, and three-year accident
histories.

The raw WES-7 dataset contained N = {N_raw:,} establishments distributed across
{n_ind} KSIC 2-digit industrial sectors. Data completeness for the 15 core safety
climate items used in this analysis was {df[SC_ALL].notna().mean().mean()*100:.1f}%,
reflecting the multi-stage completion checks applied during WES fieldwork.

## 2.2 Study Variables

### 2.2.1 Safety Culture (Primary Exposure)

The primary exposure was an establishment-level safety culture composite
constructed from 15 behavioral items organized into five theoretically grounded
dimensions. Dimensional structure was derived from contemporary safety climate
frameworks (Guldenmund, 2000; Flin et al., 2000) and validated via confirmatory
factor analysis (see Section 2.6.1). Each item was rated on a 5-point Likert scale
(1 = "strongly disagree" to 5 = "strongly agree"), with higher values indicating
a more positive safety culture.

The five dimensions were specified a priori as follows:

**Management Commitment** (3 items; Zohar, 1980) captured the degree to which
senior management prioritizes, values, and emphasizes occupational safety:
management emphasis on safety, prioritization of safety over production, and
articulated safety values.

**Safety Communication** (5 items; Flin et al., 2000) measured the openness,
frequency, and effectiveness of safety-related dialogue between workers and
management, including opportunities for safety discussion, open communication,
regular feedback, suggestion systems, and responsiveness to worker suggestions.

**Safety Training** (2 items; Burke et al., 2006) captured access to safety
training opportunities and perceived training effectiveness.

**Safety Systems and Procedures** (3 items; Hale and Hovden, 1998) measured
the presence, effectiveness, and supporting infrastructure of formal safety
management systems, including documented procedures, procedural effectiveness,
and availability of personal protective equipment.

**Worker Empowerment** (2 items; Simard and Marchand, 1994) captured workers'
perceived authority to refuse unsafe work and to engage in voluntary safety
behavior.

### 2.2.2 Industrial Accidents (Primary Outcome)

The primary outcome was the count of officially approved occupational injury
victims during calendar year 2024 (`vic_2024_appr`), representing cases confirmed
by the Korean Workers' Compensation Insurance (K-WCI) system. Because K-WCI
approval requires medical documentation and employer cooperation, these counts
provide a more objective measure of workplace injuries than self-reported figures
and are less susceptible to reporting bias. For secondary analyses, we additionally
examined self-reported injury counts (`vic_2024_occ`) reported by the same
establishments for the same calendar year. In the pooled WES-7 sample, {vic_24_sum:,}
officially approved injury cases were recorded across 2024, with
{vic_24_zero_pct:.1f}% of establishments reporting zero incidents.

### 2.2.3 Historical Accident Controls

To address the potential for reverse causality — whereby establishments with
a history of accidents may subsequently invest in safety culture improvements —
we constructed a historical accident control variable by summing officially
approved victim counts from the two preceding calendar years (`vic_prior` =
`vic_2022_appr` + `vic_2023_appr`). This variable was log-transformed as
`log_prior` = log(1 + vic_prior) to accommodate zero-inflation and entered as a
covariate in all regression models. Conditioning on prior accident experience
allows the association between contemporaneous safety culture and 2024 accidents
to be interpreted as the residual effect net of historical injury trajectories.

### 2.2.4 Covariates

We adjusted for a minimally sufficient set of confounders identified a priori
on the basis of prior literature and a directed acyclic graph. Establishment size
was measured as an ordinal variable with five categories (1 = 1–4 workers;
2 = 5–19; 3 = 20–49; 4 = 50–99; 5 = ≥100 workers). For exposure-time adjustment
in count regression, we mapped these categories to midpoint worker counts (2.5,
12, 34.5, 74.5, 200) and included the natural logarithm as a model offset,
yielding incidence rate ratios (IRRs) per worker. Industry was coded at the
KSIC 2-digit level ({n_ind} categories) and entered as fixed effects to absorb
sector-specific accident risk. Additional control variables captured working-time
arrangements (long working hours exceeding 52 hours per week; shift work) and
workforce composition (proportions of elderly, foreign, female, and workers with
disabilities, each measured as an ordinal category).

## 2.3 Missing Data

Missingness on the 15 safety culture items was effectively zero (100% complete),
reflecting fieldwork completion checks. For organizational structure control
variables, response codes "not applicable" and "don't know" were recoded as
missing. Accident count variables with system codes for missingness were treated
as zero, consistent with the interpretation that non-reporting represents no
recorded incident.

---

**Auto-generated from 0_extract.ipynb.** Source data SHA-256: `{sha}`.
N retained: {len(df):,} establishments.
"""

md_fp = os.path.join(PAPER_DIR, '01_methods_section_2.1-2.3.md')
with open(md_fp, 'w', encoding='utf-8') as f:
    f.write(methods_md)

print(f'\n✓ Saved CSV: {out_fp}')
print(f'✓ Saved Methods (2.1–2.3): {md_fp}')
